# EDA — Virus detections (ENA / INSDC)

Initial load of the local ENA virus detections table for exploratory analysis.

**Prerequisite:** run [`03_extract_viruses.ipynb`](03_extract_viruses.ipynb) first so `data/raw/virus/` contains `virus_detections.tsv` and `manifest.json`.

This notebook reads the dataset via `load_virus_detections`, inspects the extract manifest, pathogen / vector-group mix, Europe vs worldwide coverage, coordinate completeness, and temporal range. Deeper cleaning, H3 joins, and modelling features come later (see [`specs/03-spatial-temporal-matching-strategy.md`](../specs/03-spatial-temporal-matching-strategy.md)).

In [1]:
from __future__ import annotations

import json
import re

import pandas as pd

from vector_guard.data import (
    default_virus_raw_dir,
    find_repo_root,
    load_virus_detections,
)

REPO_ROOT = find_repo_root()
RAW_DIR = default_virus_raw_dir(REPO_ROOT)
MANIFEST_PATH = RAW_DIR / "manifest.json"

df = load_virus_detections()
df.shape

(102341, 13)

## Source manifest

Metadata written by the extract notebook (fetch time, citation, checksum, per-pathogen counts).

In [2]:
if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    print(f"fetched_at: {manifest.get('fetched_at')}")
    print(f"source_url: {manifest.get('source_url')}")
    print(f"sha256:     {manifest.get('sha256')}")
    print(f"row_count:  {manifest.get('row_count')}")
    print(f"citation:   {manifest.get('citation')}")
else:
    print(f"No manifest at {MANIFEST_PATH}")
    manifest = None
manifest

fetched_at: 2026-11-11T11:26:41.701243+00:00
source_url: https://www.ebi.ac.uk/ena/portal/api/search
sha256:     110a57dffa720ac80fa927d519350f7c05de5559d503914540352dc3c30b7822
row_count:  102341
citation:   European Nucleotide Archive (ENA), EMBL-EBI. Sequence metadata retrieved via the ENA Portal API (INSDC: ENA + GenBank + DDBJ). https://www.ebi.ac.uk/ena/portal/api/


{'source_url': 'https://www.ebi.ac.uk/ena/portal/api/search',
 'result': 'sequence',
 'fetched_at': '2026-11-11T11:26:41.701243+00:00',
 'tsv_name': 'virus_detections.tsv',
 'tsv_bytes': 11287024,
 'sha256': '110a57dffa720ac80fa927d519350f7c05de5559d503914540352dc3c30b7822',
 'citation': 'European Nucleotide Archive (ENA), EMBL-EBI. Sequence metadata retrieved via the ENA Portal API (INSDC: ENA + GenBank + DDBJ). https://www.ebi.ac.uk/ena/portal/api/',
 'fields': ['accession',
  'scientific_name',
  'tax_id',
  'country',
  'location',
  'collection_date',
  'host',
  'isolation_source',
  'sample_accession',
  'study_accession'],
 'filter': 'keep rows with non-empty country or collection_date; dedupe by accession',
 'row_count': 102341,
 'per_pathogen': {'West Nile virus': {'raw_rows': 25304,
   'kept_rows': 8359,
   'tax_ids': [11082]},
  'Usutu virus': {'raw_rows': 1934, 'kept_rows': 1838, 'tax_ids': [64286]},
  'Dengue virus': {'raw_rows': 60225, 'kept_rows': 57103, 'tax_ids': [126

## First look

In [3]:
print(f"Rows: {len(df):,}")
print(f"Columns ({len(df.columns)}):")
df.dtypes

Rows: 102,341
Columns (13):


accession           str
pathogen_name       str
vector_group        str
query_tax_id        str
scientific_name     str
tax_id              str
country             str
location            str
collection_date     str
host                str
isolation_source    str
sample_accession    str
study_accession     str
dtype: object

In [4]:
df.head().T

,0,1,2,3,4
accession,MH170264,MH819470,MW803141,MW835355,MW835358
pathogen_name,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus
vector_group,mosquito,mosquito,mosquito,mosquito,mosquito
query_tax_id,11082,11082,11082,11082,11082
scientific_name,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus
tax_id,11082,11082,11082,11082,11082
country,"USA:GA, Muscogee County",Canada,Serbia:Zemun,Italy,Italy
location,,,,,
collection_date,2002,15-Sep-2014,2020,2009,2011
host,Baeolophus bicolor,Culex sp.,Culex pipiens,magpie,carrion crow


## Pathogen and vector-group distribution

In [5]:
by_pathogen = (
    df["pathogen_name"]
    .value_counts()
    .rename("rows")
    .to_frame()
    .assign(pct=lambda x: (100 * x["rows"] / len(df)).round(2))
)
by_vector_group = (
    df["vector_group"]
    .value_counts()
    .rename("rows")
    .to_frame()
    .assign(pct=lambda x: (100 * x["rows"] / len(df)).round(2))
)

print("By pathogen_name:")
display(by_pathogen)
print("By vector_group:")
by_vector_group

By pathogen_name:


,rows,pct
pathogen_name,,
Dengue virus,57101,55.80
Bluetongue virus,13952,13.63
West Nile virus,8357,8.17
Chikungunya virus,6974,6.81
Crimean-Congo hemorrhagic fever virus,5272,5.15
Tick-borne encephalitis virus,3440,3.36
Zika virus,2244,2.19
Usutu virus,1838,1.80
Rift Valley fever virus,1698,1.66


By vector_group:


,rows,pct
vector_group,,
mosquito,78723,76.93
biting_midge,14397,14.07
tick,8712,8.51
sandfly,505,0.49


## Geographic coverage — Europe vs worldwide

ENA `country` values are typically `Country` or `Country: region, locality`. We parse the country token and flag Europe / Mediterranean basin countries relevant to VectorNet pairing (broad working list for EDA, not a formal ISO membership check).

In [6]:
EUROPE_MED_COUNTRIES = {
    "Albania",
    "Andorra",
    "Austria",
    "Belarus",
    "Belgium",
    "Bosnia and Herzegovina",
    "Bulgaria",
    "Croatia",
    "Cyprus",
    "Czech Republic",
    "Czechia",
    "Denmark",
    "Estonia",
    "Finland",
    "France",
    "Germany",
    "Greece",
    "Hungary",
    "Iceland",
    "Ireland",
    "Israel",
    "Italy",
    "Kosovo",
    "Latvia",
    "Lebanon",
    "Liechtenstein",
    "Lithuania",
    "Luxembourg",
    "Malta",
    "Moldova",
    "Monaco",
    "Montenegro",
    "Morocco",
    "Netherlands",
    "North Macedonia",
    "Norway",
    "Poland",
    "Portugal",
    "Romania",
    "Russia",
    "San Marino",
    "Serbia",
    "Slovakia",
    "Slovenia",
    "Spain",
    "Sweden",
    "Switzerland",
    "Syria",
    "Tunisia",
    "Turkey",
    "Ukraine",
    "United Kingdom",
    "Algeria",
    "Egypt",
    "Libya",
}


def parse_country_name(value: str) -> str:
    text = (value or "").strip()
    if not text:
        return ""
    return text.split(":", 1)[0].strip()


geo = df.assign(country_name=df["country"].map(parse_country_name))
geo["region_scope"] = geo["country_name"].map(
    lambda c: (
        "missing"
        if not c
        else "europe_med"
        if c in EUROPE_MED_COUNTRIES
        else "worldwide_other"
    )
)

scope_summary = (
    geo["region_scope"]
    .value_counts()
    .rename("rows")
    .to_frame()
    .assign(pct=lambda x: (100 * x["rows"] / len(geo)).round(2))
)
print("Region scope (Europe/Med vs other vs missing country):")
display(scope_summary)

print("\nTop countries overall:")
display(geo["country_name"].replace("", "(missing)").value_counts().head(20).to_frame("rows"))

print("\nTop Europe / Mediterranean countries:")
europe = geo.loc[geo["region_scope"] == "europe_med", "country_name"]
europe.value_counts().head(20).to_frame("rows")

Region scope (Europe/Med vs other vs missing country):


,rows,pct
region_scope,,
worldwide_other,82136,80.26
europe_med,19756,19.30
missing,445,0.43



Top countries overall:


,rows
country_name,
India,9050
USA,8285
Brazil,7925
China,7591
Thailand,6937
Russia,4667
Singapore,4375
Viet Nam,4337
France,2831



Top Europe / Mediterranean countries:


,rows
country_name,
Russia,4667
France,2831
Italy,2398
Israel,1263
Turkey,1234
Germany,1025
Spain,729
Netherlands,525
Greece,515


In [7]:
scope_by_pathogen = (
    geo.groupby(["pathogen_name", "region_scope"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["europe_med", "worldwide_other", "missing"], fill_value=0)
)
scope_by_pathogen["total"] = scope_by_pathogen.sum(axis=1)
scope_by_pathogen["europe_med_pct"] = (
    100 * scope_by_pathogen["europe_med"] / scope_by_pathogen["total"]
).round(2)
scope_by_pathogen.sort_values("total", ascending=False)

region_scope,europe_med,worldwide_other,missing,total,europe_med_pct
pathogen_name,,,,,
Dengue virus,451,56455,195,57101,0.79
Bluetongue virus,6847,7058,47,13952,49.08
West Nile virus,2523,5707,127,8357,30.19
Chikungunya virus,113,6813,48,6974,1.62
Crimean-Congo hemorrhagic fever virus,3694,1578,0,5272,70.07
Tick-borne encephalitis virus,3101,321,18,3440,90.15
Zika virus,65,2175,4,2244,2.90
Usutu virus,1802,35,1,1838,98.04
Rift Valley fever virus,49,1648,1,1698,2.89


## Coordinate and region completeness

H3 Tier-1 joins need usable coordinates (`location` in ENA, often `lat lon` text). Most rows only have country/region text — those fall back to country-level Tier 2 in the matching strategy.

In [8]:
_COORD_RE = re.compile(
    r"(?P<lat>-?\d+(?:\.\d+)?)\s*[NnSs]?\s+[NnSs]?\s*(?P<lon>-?\d+(?:\.\d+)?)\s*[EeWw]?",
    re.IGNORECASE,
)

def has_coord_like_location(value: str) -> bool:
    text = (value or "").strip()
    if not text:
        return False
    return _COORD_RE.search(text) is not None


geo["has_country"] = geo["country_name"] != ""
geo["has_subnational"] = geo["country"].str.contains(":", regex=False, na=False)
geo["has_location"] = geo["location"].fillna("").str.strip() != ""
geo["has_coords"] = geo["location"].map(has_coord_like_location)

completeness = pd.Series(
    {
        "has_country": int(geo["has_country"].sum()),
        "has_subnational_in_country": int(geo["has_subnational"].sum()),
        "has_location_field": int(geo["has_location"].sum()),
        "has_coord_like_location": int(geo["has_coords"].sum()),
    },
    name="rows",
).to_frame()
completeness["pct"] = (100 * completeness["rows"] / len(geo)).round(2)

print("Field completeness:")
display(completeness)

print("\nCoord-like location rate by region scope:")
(
    geo.groupby("region_scope")["has_coords"]
    .agg(rows="size", with_coords="sum")
    .assign(pct=lambda x: (100 * x["with_coords"] / x["rows"]).round(2))
)

Field completeness:


,rows,pct
has_country,101892,99.57
has_subnational_in_country,29720,29.04
has_location_field,2656,2.60
has_coord_like_location,2656,2.60



Coord-like location rate by region scope:


,rows,with_coords,pct
region_scope,,,
europe_med,19756,155,0.78
missing,445,191,42.92
worldwide_other,82136,2310,2.81


## Temporal range

Parse a coarse year from `collection_date` (ENA dates vary: `2003`, `02-Aug-2005`, ranges, etc.).

In [9]:
def parse_year(value: str) -> int | None:
    text = (value or "").strip()
    if not text:
        return None
    match = re.search(r"(19|20)\d{2}", text)
    if not match:
        return None
    year = int(match.group(0))
    if 1900 <= year <= 2100:
        return year
    return None


geo["year"] = geo["collection_date"].map(parse_year)
with_year = geo["year"].notna().sum()
print(f"Rows with parseable year: {with_year:,} ({100 * with_year / len(geo):.1f}%)")

years = geo["year"].dropna().astype(int)
if len(years):
    print(f"Year range: {years.min()} – {years.max()}")
    print("\nRows by decade:")
    decade = (years // 10 * 10).value_counts().sort_index()
    display(decade.to_frame("rows"))

print("\nYear coverage by pathogen (min / max / n):")
(
    geo.dropna(subset=["year"])
    .groupby("pathogen_name")["year"]
    .agg(min_year="min", max_year="max", n="count")
    .astype({"min_year": int, "max_year": int})
    .sort_values("n", ascending=False)
)

Rows with parseable year: 95,764 (93.6%)
Year range: 1905 – 2026

Rows by decade:


,rows
year,
1900,2
1930,36
1940,133
1950,306
1960,412
1970,613
1980,1401
1990,2786
2000,19351



Year coverage by pathogen (min / max / n):


,min_year,max_year,n
pathogen_name,,,
Dengue virus,1905,2026,54135
Bluetongue virus,1937,2026,12268
West Nile virus,1931,2026,7804
Chikungunya virus,1952,2026,6500
Crimean-Congo hemorrhagic fever virus,1956,2025,4983
Tick-borne encephalitis virus,1937,2025,3056
Zika virus,1947,2026,2193
Usutu virus,1959,2025,1817
Rift Valley fever virus,1944,2023,1657


## Quick takeaways

- The extract is worldwide sequence metadata; Europe/Med is only a subset of rows.
- Exact coordinates (`location`) are sparse — most modelling joins will start at **country Tier 2**, with H3 Tier 1 only where coords exist.
- Pathogen mix is imbalanced (e.g. dengue dominates); class balance needs care before training.
- Next steps: deeper cleaning, ISO-2 normalization, H3 assignment where possible, and VectorNet join per the matching strategy spec.